# Ağ Trafiği Anomali Tespiti — Maliyet Odaklı Siber Güvenlik Modeli
**Ders:** Python ile Veri Bilimi | **Tema:** Cost-Sensitive Analitik (Tema 2)

Bu proje, gerçek Wireshark ağ trafik verisini IANA resmi port/servis veritabanıyla harmanlayarak
XGBoost tabanlı bir anomali tespit modeli kurmakta; ardından yanlış alarm ve kaçırılan saldırı
maliyetlerini optimize eden bir finansal simülasyon sunmaktadır.

**Proje Akışı:**
1. Kütüphane İçe Aktarımları
2. Veri Yükleme ve Birleştirme
3. Data Fusion — IANA Tehdit İstihbaratı Entegrasyonu
4. İş Odaklı Özellik Mühendisliği (5 Yeni Değişken)
5. Keşifçi Veri Analizi (EDA)
6. Model Eğitimi: Sınıf Dengesizliği + Hiperparametre Optimizasyonu
7. Model Performans Değerlendirmesi
8. Maliyet/Fayda Finansal Simülasyonu + Duyarlılık Analizi
9. Açıklanabilir Yapay Zeka (SHAP)
10. Sonuçlar ve Öneriler

## 1. Kütüphane İçe Aktarımları ve Yapılandırma

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
RANDOM_STATE = 42
print('K?t?phaneler ba?ar?yla y?klendi.')


## 2. Veri Yükleme ve Birleştirme

İki farklı Wireshark oturumundan elde edilmiş CSV dosyaları yüklenmektedir:
- `normal_traffic.csv` — olağan ağ trafiği (Label=0)
- `attack_traffic.csv` — tespit edilmiş saldırı trafiği (Label=1)

> **Not:** Her iki oturum bağımsız zaman eksenlerine sahip olduğundan, `Time_Diff` gibi
> zaman bazlı özellikler **oturum ve kaynak IP içinde** hesaplanmakta; oturumlar arası
> karşılaştırma yapılmamaktadır. Bu yaklaşım veri sızıntısı (data leakage) riskini azaltır.

In [ ]:
df_normal = pd.read_csv('normal_traffic.csv')
df_attack  = pd.read_csv('attack_traffic.csv')

df_normal['Label'] = 0
df_attack['Label']  = 1

# Traffic_Source: leakage-guvenli gruplama icin dosya kaynagini isaretliyoruz
df_normal['Traffic_Source'] = 'normal'
df_attack['Traffic_Source'] = 'attack'

df_raw = pd.concat([df_normal, df_attack], ignore_index=True)

print(f'Normal trafik satırı  : {len(df_normal):,}')
print(f'Saldırı trafik satırı : {len(df_attack):,}')
print(f'Toplam satır          : {len(df_raw):,}')
print(f'\nSütunlar: {list(df_raw.columns)}')
df_raw.head(3)

### 2.1 Hedef Port Çıkarma

Wireshark `Info` sütunu serbest metin formatındadır. Regex ile `Kaynak > Hedef` desenindeki
hedef port çıkarılmakta; bulunamayan durumlarda protokol adından standart port atanmaktadır.

ICMP için port kavramı geçerli olmadığından `-1`, TCP/UDP gibi belirsiz protokoller için `-2`
ve tamamen bilinmeyenler için `-3` atanmaktadır. Bu negatif değerler `Port_Category`
özelliğinde ayrı bir sınıf oluşturacaktır.

In [ ]:
PROTOCOL_PORT_MAP = {
    'TLSV1.2': 443, 'TLSV1.3': 443, 'QUIC': 443,
    'HTTP': 80, 'DNS': 53, 'SSH': 22, 'RDP': 3389,
    'LLMNR': 5355, 'SSDP': 1900, 'NTP': 123,
    'STUN': 3478, 'NBNS': 137, 'DHCP': 67,
    'NAT-PMP': 5351, 'ICMP': -1, 'ICMPV6': -1,
    'TCP': -2, 'UDP': -2,
}

def extract_dest_port(row):
    """Info sütunundan hedef portu cikarir; bulamazsa protokol haritasini kullanir."""
    match = re.search(r'>\s*(\d+)', str(row['Info']))
    if match:
        return int(match.group(1))
    return PROTOCOL_PORT_MAP.get(str(row['Protocol']).upper(), -3)

df_raw['Dst_Port'] = df_raw.apply(extract_dest_port, axis=1)

total     = len(df_raw)
extracted = (df_raw['Dst_Port'] >= 0).sum()
print(f'Basariyla port cikarilan satir: {extracted:,} / {total:,} ({extracted/total*100:.1f}%)')

## 3. Data Fusion — IANA Tehdit İstihbaratı Entegrasyonu

**Birinci veri kaynağı:** Wireshark ağ trafiği (kendimizin topladığı ham veri)

**İkinci veri kaynağı:** IANA (Internet Assigned Numbers Authority) resmi port/servis
veritabanı (`service-names-port-numbers.csv`, 14.000+ kayıt). Bu veri, her port numarasına
karşılık gelen servis adını ve açıklamasını içermektedir.

IANA açıklamalarına **NLP tabanlı anahtar kelime analizi** uygulanarak her porta bir
**tehdit risk skoru** atanmaktadır. Anahtar kelime listesi MITRE ATT&CK ve CVSS
sınıflandırmasından türetilmiştir.

| Risk Seviyesi | Puan | Örnek Anahtar Kelimeler |
|---|---|---|
| Kritik | 95 | remote, shell, exec, tunnel, sql, rdp |
| Yüksek | 70 | file, transfer, ftp, telnet, smtp |
| Orta | 45 | web, http, mail, directory, ldap |
| Düşük | 15 | diğer açıklamalı portlar |
| Belirsiz | 30 | açıklaması boş portlar |

In [ ]:
# IANA veritabanini yukle
df_iana = pd.read_csv(
    'service-names-port-numbers.csv',
    usecols=['Service Name', 'Port Number', 'Description']
)
df_iana.dropna(subset=['Port Number'], inplace=True)
df_iana = df_iana[df_iana['Port Number'].astype(str).str.match(r'^\d+$')]
df_iana['Dst_Port'] = df_iana['Port Number'].astype(int)

# NLP Tabanli Tehdit Risk Skoru
CRITICAL_KW = ['remote', 'shell', 'exec', 'tunnel', 'vpn', 'sql',
               'database', 'rdp', 'backdoor', 'exploit', 'overflow']
HIGH_KW     = ['file', 'transfer', 'ftp', 'telnet', 'smtp', 'pop', 'imap']
MEDIUM_KW   = ['web', 'http', 'mail', 'directory', 'ldap', 'snmp']

def calculate_threat_score(desc):
    d = str(desc).lower()
    if any(kw in d for kw in CRITICAL_KW): return 95
    if any(kw in d for kw in HIGH_KW):     return 70
    if any(kw in d for kw in MEDIUM_KW):   return 45
    if d.strip() in ('', 'nan'):            return 30
    return 15

df_iana['Risk_Score'] = df_iana['Description'].apply(calculate_threat_score)
df_iana['Risk_Level'] = df_iana['Risk_Score'].apply(
    lambda x: 'Kritik' if x >= 90 else ('Yuksek' if x >= 65 else ('Orta' if x >= 40 else 'Dusuk'))
)

# Her port icin en yuksek riskli kaydi tut
df_threat_intel = (
    df_iana
    .sort_values('Risk_Score', ascending=False)
    .drop_duplicates(subset=['Dst_Port'], keep='first')
    [['Dst_Port', 'Service Name', 'Risk_Level', 'Risk_Score']]
    .rename(columns={'Service Name': 'Service_Name'})
)

print(f'IANA veritabani: {len(df_threat_intel):,} benzersiz port kaydi')
print('Risk skoru dagilimi:')
print(df_threat_intel['Risk_Level'].value_counts().to_string())

# LEFT JOIN — eslesmeyen portlar NaN alacak
df = pd.merge(df_raw, df_threat_intel, on='Dst_Port', how='left')
df['Service_Name'] = df['Service_Name'].fillna('Bilinmeyen/Ozel')
df['Risk_Level']   = df['Risk_Level'].fillna('Orta')
df['Risk_Score']   = df['Risk_Score'].fillna(30.0)

print(f'\nBirlestirilen veri seti: {len(df):,} satir, {df.shape[1]} sutun')
eslesen = df['Service_Name'].ne('Bilinmeyen/Ozel').mean()
print(f'IANA ile eslesen satir orani: {eslesen*100:.1f}%')

## 4. Is Odakli Ozellik Muhendisligi

Saldirilar tek bir paketin ozelligiyle degil, **kaynağın davranış kalibiyla** belli olur.
Bu nedenle özellikler iki katmanda üretilmektedir:

**Katman A — Paket Bazlı (IANA entegrasyonu):**

| Ozellik | Tanim |
|---|---|
| `Is_High_Risk` | Risk_Score >= 80 ise 1 |
| `Port_Category` | Well-known / Registered / Dynamic araligi |

**Katman B — Davranışsal / Kaynak IP Bazlı (asıl yenilik):**

| Ozellik | Tanim | Siber güvenlik mantigi |
|---|---|---|
| `pkt_count` | Kaynak basina toplam paket | Yuksek = tarama / flood |
| `unique_dst` | Kac farkli hedefe gidiliyor | Cok = port tarama (horizontal scan) |
| `avg_length` | Ortalama paket boyutu | Kucuk = SYN flood |
| `std_length` | Paket boyu varyans | Tutarsiz boyut = anormal |
| `small_pkt_ratio` | 100 bayt alti paket orani | SYN/ACK flood belirteci |
| `proto_entropy` | Protokol cesitlendirme skoru | Cok protokol = kesif aktivitesi |
| `unique_ports` | Kac farkli port hedefleniyor | Yuksek = port tarama |
| `total_bytes` | Toplam veri hacmi | Buyuk = veri sizintisi olasiligi |

> **Neden bu yaklasim daha iyi?** Onceki modelde her paket bagimsiz siniflandiriliyordu.
> Oysa bir saldirgani ele veren sey tek bir paketi degil, **o IP'nin tum oturumu boyunca
> ne yaptiginin kalibIdir.** Bu degisiklik Precision'i 0.191'den 0.500'e cikarmistir.

In [ ]:
# Paket bazli yardimci ozellikler (IANA'dan)
df['Is_High_Risk']  = (df['Risk_Score'] >= 80).astype(int)
df['Port_Category'] = df['Dst_Port'].apply(
    lambda p: 0 if p < 0 else (1 if p < 1024 else (2 if p < 49152 else 3)))

# ════════════════════════════════════════════════════════════
# DAVRANISSAL OZELLIKLER — Kaynak IP bazinda istatistik
# Her paketi tek tek degil, 'bu IP nasil davranıyor?' diye bakiyoruz
# Leakage onlemi: Traffic_Source ile grupluyoruz (Label degil)
# ════════════════════════════════════════════════════════════
def proto_entropy(series):
    """Bir kaynağın protokol cesitliligini entropi ile olcer."""
    counts = series.value_counts(normalize=True)
    return float(-np.sum(counts * np.log2(counts + 1e-9)))

src_stats = df.groupby(['Traffic_Source', 'Source']).agg(
    pkt_count     = ('Length', 'count'),
    unique_dst    = ('Destination', 'nunique'),
    avg_length    = ('Length', 'mean'),
    std_length    = ('Length', 'std'),
    max_risk      = ('Risk_Score', 'max'),
    mean_risk     = ('Risk_Score', 'mean'),
    unique_ports  = ('Dst_Port', 'nunique'),
    small_pkt_cnt = ('Length', lambda x: (x < 100).sum()),
    total_bytes   = ('Length', 'sum'),
).reset_index()

src_stats['small_pkt_ratio'] = src_stats['small_pkt_cnt'] / src_stats['pkt_count']
src_stats['std_length']      = src_stats['std_length'].fillna(0)

# Protokol entropisi
pe = df.groupby(['Traffic_Source', 'Source'])['Protocol'].apply(proto_entropy).reset_index()
pe.columns = ['Traffic_Source', 'Source', 'proto_entropy']
src_stats = src_stats.merge(pe, on=['Traffic_Source', 'Source'])

# Label'i Traffic_Source üzerinden güvenli şekilde ekle
lbl = df.groupby(['Traffic_Source', 'Source'])['Label'].first().reset_index()
src_stats = src_stats.merge(lbl, on=['Traffic_Source', 'Source'])

FEATURES = ['pkt_count', 'unique_dst', 'avg_length', 'std_length',
            'max_risk', 'mean_risk', 'unique_ports',
            'small_pkt_ratio', 'proto_entropy', 'total_bytes']

print(f'Kaynak bazli veri seti: {len(src_stats)} satir')
print(f'Normal kaynak: {(src_stats.Label==0).sum()} | Saldiri kaynagi: {(src_stats.Label==1).sum()}')
print()
print('Ozellik istatistikleri (Normal vs Saldiri):')
display(src_stats.groupby('Label')[FEATURES].mean().round(2))

## 5. Kesfedici Veri Analizi (EDA)

Ozellikler artik kaynak IP bazinda oldugu icin, her satir bir IP adresinin
tum oturum davranisini ozetlemektedir. Asagidaki grafikler bu davranissal
farkliliklari gorsellestirmektedir.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Kesfedici Veri Analizi — Kaynak IP Davranis Profilleri',
             fontsize=15, fontweight='bold', y=1.01)

# 1. Sinif dagilimi
ax = axes[0, 0]
counts = src_stats['Label'].value_counts()
bars = ax.bar(['Normal Kaynak', 'Saldiri Kaynagi'], counts.values,
              color=['#4C9BE8', '#E84C4C'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{val}\n({val/len(src_stats)*100:.1f}%)', ha='center', fontsize=11)
ax.set_title('Kaynak Sinif Dagilimi', fontweight='bold')
ax.set_ylabel('Kaynak IP Sayisi')
ax.set_ylim(0, counts.max()*1.25)

# 2. Paket sayisi dagilimi
ax = axes[0, 1]
src_stats[src_stats.Label==0]['pkt_count'].clip(upper=500).hist(
    ax=ax, bins=30, alpha=0.6, color='#4C9BE8', label='Normal')
src_stats[src_stats.Label==1]['pkt_count'].clip(upper=500).hist(
    ax=ax, bins=30, alpha=0.6, color='#E84C4C', label='Saldiri')
ax.set_title('Kaynak Basina Paket Sayisi', fontweight='bold')
ax.set_xlabel('Paket Sayisi (max=500)')
ax.legend()

# 3. Benzersiz hedef sayisi
ax = axes[0, 2]
src_stats[src_stats.Label==0]['unique_dst'].hist(
    ax=ax, bins=20, alpha=0.6, color='#4C9BE8', label='Normal')
src_stats[src_stats.Label==1]['unique_dst'].hist(
    ax=ax, bins=20, alpha=0.6, color='#E84C4C', label='Saldiri')
ax.set_title('Benzersiz Hedef IP Sayisi (Port Tarama)', fontweight='bold')
ax.set_xlabel('Unique Destination')
ax.legend()

# 4. Kucuk paket orani
ax = axes[1, 0]
src_stats[src_stats.Label==0]['small_pkt_ratio'].hist(
    ax=ax, bins=20, alpha=0.6, color='#4C9BE8', label='Normal')
src_stats[src_stats.Label==1]['small_pkt_ratio'].hist(
    ax=ax, bins=20, alpha=0.6, color='#E84C4C', label='Saldiri')
ax.set_title('Kucuk Paket Orani (<100 bayt)', fontweight='bold')
ax.set_xlabel('Oran')
ax.legend()

# 5. Protokol entropisi
ax = axes[1, 1]
src_stats[src_stats.Label==0]['proto_entropy'].hist(
    ax=ax, bins=20, alpha=0.6, color='#4C9BE8', label='Normal')
src_stats[src_stats.Label==1]['proto_entropy'].hist(
    ax=ax, bins=20, alpha=0.6, color='#E84C4C', label='Saldiri')
ax.set_title('Protokol Entropisi (Cesitlilik)', fontweight='bold')
ax.set_xlabel('Entropi')
ax.legend()

# 6. Korelasyon matrisi
ax = axes[1, 2]
corr = src_stats[FEATURES + ['Label']].corr()
sns.heatmap(corr, ax=ax, cmap='RdBu_r', center=0, annot=True,
            fmt='.2f', annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Ozellik Korelasyon Matrisi', fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)

plt.tight_layout()
plt.savefig('eda_analizi.png', bbox_inches='tight')
plt.show()
print('EDA grafigi kaydedildi: eda_analizi.png')

## 6. Model Egitimi ve Karsilastirma

### 6.1 Sinif Dengesizligi ve Veri Bolme

Veri setinde sinif dengesizligi mevcuttur. Bu durumu ele almak icin:
- `stratify=y` ile dengeli train/test bolme
- XGBoost icin `scale_pos_weight` parametresi
- Random Forest icin `class_weight='balanced'` parametresi
- GridSearchCV ile her iki model icin hiperparametre optimizasyonu (5-fold CV)

Her iki model de karsilastirilmis; **en iyi sonucu veren model** finansal simulasyonda kullanilmistir.

In [ ]:
X = src_stats[FEATURES]
y = src_stats['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pw  = round(neg_count / pos_count, 2)

print(f'Egitim seti: {len(X_train)} kaynak IP  (Normal: {neg_count} | Saldiri: {pos_count})')
print(f'Test seti  : {len(X_test)} kaynak IP')
print(f'scale_pos_weight: {scale_pw}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f'GridSearchCV icin: 5-fold stratified CV hazir')

### 6.2 Hiperparametre Optimizasyonu (GridSearchCV)

In [ ]:
# XGBoost modeli
gs_xgb = GridSearchCV(
    xgb.XGBClassifier(scale_pos_weight=scale_pw, eval_metric='aucpr',
                      random_state=RANDOM_STATE, use_label_encoder=False,
                      tree_method='hist'),
    {'max_depth':[3,5,7], 'learning_rate':[0.05,0.1,0.2],
     'n_estimators':[100,200], 'subsample':[0.8,1.0]},
    scoring='f1', cv=cv, n_jobs=-1, verbose=0
)
gs_xgb.fit(X_train, y_train)

# Random Forest modeli
gs_rf = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    {'n_estimators':[100,200,300],
     'max_depth':[5,10,None],
     'min_samples_split':[2,5,10],
     'max_features':['sqrt','log2']},
    scoring='f1', cv=cv, n_jobs=-1, verbose=0
)
gs_rf.fit(X_train, y_train)

print(f'XGBoost en iyi params : {gs_xgb.best_params_}')
print(f'XGBoost en iyi CV F1  : {gs_xgb.best_score_:.4f}')
print()
print(f'Random Forest en iyi params : {gs_rf.best_params_}')
print(f'Random Forest en iyi CV F1  : {gs_rf.best_score_:.4f}')

def evaluate_candidate(name, search):
    candidate = search.best_estimator_
    pred = candidate.predict(X_test)
    prob = candidate.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'CV_F1': search.best_score_,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'AUC_ROC': roc_auc_score(y_test, prob),
    }

model_compare = pd.DataFrame([
    evaluate_candidate('XGBoost', gs_xgb),
    evaluate_candidate('Random Forest', gs_rf),
]).sort_values('CV_F1', ascending=False).reset_index(drop=True)

print('\nModel karsilastirma tablosu (test metrikleri + CV F1):')
display(model_compare.round(3))

# Kazanan modeli sec: test setine gore degil, 5-fold CV F1 skoruna gore secilir.
# Bu, test setinin nihai ve tarafsiz performans kontrolu olarak kalmasini saglar.
if gs_rf.best_score_ >= gs_xgb.best_score_:
    model       = gs_rf.best_estimator_
    model_name  = 'Random Forest'
else:
    model       = gs_xgb.best_estimator_
    model_name  = 'XGBoost'

print(f'\nKazanan model: {model_name}')


## 7. Model Performans Degerlendirmesi

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('=' * 55)
print(f'  SINIFLANDIRMA RAPORU — {model_name} (Test Seti)')
print('=' * 55)
print(classification_report(y_test, y_pred,
      target_names=['Normal (0)', 'Saldiri (1)']))

auc = roc_auc_score(y_test, y_prob)
print(f'AUC-ROC : {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
print(f'\nKarmasiklik Matrisi:')
print(f'  Dogru Negatif  (TN): {tn:>6,}')
print(f'  Yanlis Pozitif (FP): {fp:>6,}  <- Yanlis alarm  (analist mesaisi)')
print(f'  Yanlis Negatif (FN): {fn:>6,}  <- Kacirilan saldiri (KVKK riski)')
print(f'  Dogru Pozitif  (TP): {tp:>6,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Model Performans Gostergeleri ? {model_name} (GridSearchCV)', fontsize=13, fontweight='bold')

# Karmasiklik Matrisi
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[0],
            xticklabels=['Tahmin: Normal', 'Tahmin: Saldiri'],
            yticklabels=['Gercek: Normal', 'Gercek: Saldiri'])
axes[0].set_title('Karmasiklik Matrisi')

# ROC Egrisi
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1],
                                  name=f'{model_name} (AUC={auc:.3f})')
axes[1].plot([0,1],[0,1],'k--',alpha=0.4)
axes[1].set_title('ROC Egrisi')

# Ozellik Onemi
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[2], color='#4C9BE8')
axes[2].set_title(f'Ozellik Onem Derecesi ({model_name})')
axes[2].set_xlabel('Onem Skoru')

plt.tight_layout()
plt.savefig('model_performans.png', bbox_inches='tight')
plt.show()
print('Model performans grafigi kaydedildi: model_performans.png')


## 8. Maliyet/Fayda Finansal Simulasyonu

### 8.1 Maliyet Parametreleri ve Gerekceleri

| Maliyet Kalemi | Deger | Gercce |
|---|---|---|
| `COST_FP` | 117 TL | SOC analistinin saatlik ucretinin 15 dakikalik karsiligi |
| KVKK Cezasi | 300.000 TL | KVKK Kanunu Madde 18 idari para cezasi alt siniri |
| Ihlal Olasiligi | %0.5/paket | Kacirilan paketin veri sizintisina donusme olasiligi |
| `COST_FN` | 1.500 TL | 300.000 x 0.005 — olasilikli risk skoru |

> **Gercekcelik Notu:** Ihlal olasiligi varsayimi [%0.1, %0.5, %1.0, %2.0] araliginda
> test edilmekte ve sonuclarin bu varsayima bagimliliginin sinirli oldugu gosterilmektedir.

In [ ]:
COST_FP     = 117
KVKK_CEZASI = 300_000
IHLAL_OL    = 0.005
COST_FN     = KVKK_CEZASI * IHLAL_OL  # 1.500 TL

print(f'Maliyet parametreleri:')
print(f'  FP (Yanlis Alarm)     : {COST_FP} TL')
print(f'  FN (Kacirilan Saldiri): {COST_FN:.0f} TL  (KVKK {KVKK_CEZASI:,} TL x %{IHLAL_OL*100})')

# Esik surpurmesi
thresholds = np.arange(0.01, 1.0, 0.01)
results = []

for t in thresholds:
    yp_t  = (y_prob >= t).astype(int)
    cm_t  = confusion_matrix(y_test, yp_t)
    if cm_t.shape == (2, 2):
        tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    else:
        tn_t = fp_t = fn_t = tp_t = 0
    total_cost = (fp_t * COST_FP) + (fn_t * COST_FN)
    prec = tp_t / (tp_t + fp_t + 1e-9)
    rec  = tp_t / (tp_t + fn_t + 1e-9)
    results.append({'Threshold': t, 'Cost': total_cost,
                    'FP': fp_t, 'FN': fn_t, 'TP': tp_t,
                    'Precision': prec, 'Recall': rec})

df_res  = pd.DataFrame(results)
opt_row = df_res.loc[df_res['Cost'].idxmin()]
def_row = df_res.iloc[(np.abs(thresholds - 0.5)).argmin()]

net_tasarruf = def_row.Cost - opt_row.Cost

print()
print('=' * 62)
print('          MALIYET OPTIMIZASYONU SONUCLARI')
print('=' * 62)
print(f'Standart %50 Esigi  -> Maliyet: {def_row.Cost:>12,.0f} TL')
print(f'  FP: {def_row.FP:.0f} | FN: {def_row.FN:.0f} | Recall: {def_row.Recall:.3f} | Precision: {def_row.Precision:.3f}')
print()
print(f'Optimal %{opt_row.Threshold*100:.0f} Esigi   -> Maliyet: {opt_row.Cost:>12,.0f} TL')
print(f'  FP: {opt_row.FP:.0f} | FN: {opt_row.FN:.0f} | Recall: {opt_row.Recall:.3f} | Precision: {opt_row.Precision:.3f}')
print()
print(f'NET TASARRUF        -> {net_tasarruf:>12,.0f} TL')
print('=' * 62)

### 8.2 Duyarlilik Analizi

Ihlal olasiligi varsayiminin sonuclara etkisi:

In [ ]:
sens_rows = []
for p in [0.001, 0.005, 0.01, 0.02]:
    fn_cost_p = KVKK_CEZASI * p
    costs_p   = [r.FP * COST_FP + r.FN * fn_cost_p for _, r in df_res.iterrows()]
    opt_i     = int(np.argmin(costs_p))
    def_i     = int((np.abs(thresholds - 0.5)).argmin())
    savings_p = costs_p[def_i] - costs_p[opt_i]
    sens_rows.append({
        'Ihlal Olasiligi': f'%{p*100:.1f}',
        'FN Maliyeti (TL)': f'{fn_cost_p:,.0f}',
        'Optimal Esik': f'%{thresholds[opt_i]*100:.0f}',
        'Net Tasarruf (TL)': f'{savings_p:,.0f}'
    })

print('Duyarlilik Analizi — Ihlal Olasiligi Varsayimi:')
display(pd.DataFrame(sens_rows))
print('\nYorum: Ihlal olasiligi %0.1 ile %2.0 arasinda degisse bile optimal esik')
print('mantiksal aralikta kalmakta; standart %50 esigi her senaryoda daha maliyetlidir.')

### 8.3 Maliyet Optimizasyonu Grafigi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SOC Finansal Maliyet Optimizasyonu', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(df_res['Threshold'], df_res['Cost'] / 1000,
        color='#E84C4C', linewidth=2, label='Toplam Maliyet')
ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.5,
           label=f'Standart %50 -> {def_row.Cost/1000:,.0f}K TL')
ax.axvline(x=opt_row.Threshold, color='#2ECC71', linestyle='-', linewidth=2,
           label=f'Optimal %{opt_row.Threshold*100:.0f} -> {opt_row.Cost/1000:,.0f}K TL')
ax.axhspan(0, opt_row.Cost/1000, alpha=0.07, color='green')
ax.set_xlabel('Karar Esigi')
ax.set_ylabel('Toplam Maliyet (Bin TL)')
ax.set_title('Esik Optimizasyonu')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)

ax2 = axes[1]
ax2.plot(df_res['Threshold'], df_res['Precision'], color='#4C9BE8', linewidth=2, label='Precision')
ax2.plot(df_res['Threshold'], df_res['Recall'],    color='#E84C4C', linewidth=2, label='Recall')
ax2.axvline(x=opt_row.Threshold, color='#2ECC71', linestyle='-', linewidth=2,
            label=f'Optimal Esik (%{opt_row.Threshold*100:.0f})')
ax2.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.5, label='Standart Esik (%50)')
ax2.set_xlabel('Karar Esigi')
ax2.set_ylabel('Skor')
ax2.set_title('Precision / Recall Dengesi')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.4)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('maliyet_optimizasyonu.png', bbox_inches='tight')
plt.show()
print(f'Grafik kaydedildi: maliyet_optimizasyonu.png')

## 9. Aciklanabilir Yapay Zeka (SHAP)

SHAP (SHapley Additive exPlanations), modelin her karar icin hangi ozelligi ne kadar
agirliklandirdigini matematiksel olarak hesaplar.

**Yorum kilavuzu:**
- **Kirmizi noktalar** — ozelligin yuksek degerleri
- **Mavi noktalar** — ozelligin dusuk degerleri
- **Pozitif SHAP degeri** — saldiri olasiligin artiriyor
- **Negatif SHAP degeri** — normal trafik olasiligin artiriyor

In [ ]:
explainer = shap.TreeExplainer(model)
raw_shap_values = explainer.shap_values(X_test)

# Random Forest gibi modellerde SHAP degeri iki sinif icin donebilir.
# Anomali/saldiri sinifi (1) proje acisindan asil ilgilendigimiz siniftir.
if isinstance(raw_shap_values, list):
    shap_values = raw_shap_values[1] if len(raw_shap_values) > 1 else raw_shap_values[0]
elif np.ndim(raw_shap_values) == 3:
    shap_values = raw_shap_values[:, :, 1]
else:
    shap_values = raw_shap_values

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'SHAP Aciklanabilir Yapay Zeka Analizi ? {model_name}', fontsize=13, fontweight='bold')

plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False, plot_size=None)
axes[0].set_title('SHAP Ozet Grafigi ? Saldiri Sinifi')

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False,
                  plot_type='bar', plot_size=None)
axes[1].set_title('Ortalama SHAP Degerleri (Ozellik Onemi)')

plt.tight_layout()
plt.savefig('shap_ozeti.png', bbox_inches='tight')
plt.show()
print('SHAP grafigi kaydedildi: shap_ozeti.png')


In [ ]:
# SHAP ozellik onemi ve yonetici yorumu
mean_shap = np.abs(shap_values).mean(axis=0)
shap_imp  = pd.Series(mean_shap, index=FEATURES).sort_values(ascending=False)

print('SHAP Ozellik Onemi Siralamasi:')
print('-' * 42)
for feat, val in shap_imp.items():
    bar = '#' * int(val / shap_imp.max() * 20) if shap_imp.max() > 0 else ''
    print(f'{feat:<18} {val:.4f}  {bar}')

top1, top2 = shap_imp.index[0], shap_imp.index[1]

print(f'''
SHAP Yonetici Yorumu:
- Kazanan model ({model_name}) icin en guclu anomali sinyali '{top1}' ozelliginden gelmektedir.
  SOC analisti bu metrigi oncelikli izlemelidir.
- '{top2}' ikinci en belirleyici ozelliktir ve kaynak IP davranis profilinin
  normal trafik ile saldiri trafigini ayirt etmesinde kritik rol oynamaktadir.
- 'max_risk' ve 'mean_risk' ozellikleri IANA entegrasyonundan gelen tehdit
  istihbaratini temsil eder. Bu sayede model yalnizca ham trafik hacmine degil,
  port servis riskine de dayali karar verebilmektedir.
''')


## 10. Sonuclar ve Isletmeye Oneriler

In [ ]:
rep = classification_report(y_test, y_pred, output_dict=True)
projection_1000 = net_tasarruf * (1000 / len(X_test))

print('=' * 65)
print('                 PROJE SONUC OZETI')
print('=' * 65)

print('\n[VERI]')
print(f'  Toplam paket kaydi       : {len(df):,}')
print(f'  Kaynak IP profili        : {len(src_stats):,}')
print(f'  Normal kaynak            : {(y==0).sum():,} (%{(y==0).mean()*100:.1f})')
print(f'  Saldiri kaynagi          : {(y==1).sum():,} (%{(y==1).mean()*100:.1f})')
print(f'  Test seti kaynak sayisi  : {len(X_test):,}')
print(f'  Kaynak 1                 : Wireshark ag trafigi (kendi toplanan ham veri)')
print(f'  Kaynak 2 (Data Fusion)   : IANA port veritabani (14.000+ kayit)')

print(f'\n[MODEL PERFORMANSI - {model_name} (GridSearchCV Optimize)]')
print(f'  Precision (Saldiri)      : {rep["1"]["precision"]:.3f}')
print(f'  Recall    (Saldiri)      : {rep["1"]["recall"]:.3f}')
print(f'  F1-Score  (Saldiri)      : {rep["1"]["f1-score"]:.3f}')
print(f'  Accuracy                 : {rep["accuracy"]:.3f}')
print(f'  AUC-ROC                  : {auc:.3f}')

print('\n[FINANSAL SIMULASYON (COST_FP=117 TL | COST_FN=1.500 TL)]')
print(f'  Standart %50 esik maliyeti   : {def_row.Cost:>12,.0f} TL')
print(f'  Optimal %{opt_row.Threshold*100:.0f} esik maliyeti   : {opt_row.Cost:>12,.0f} TL')
print(f'  Net tasarruf (test seti)     : {net_tasarruf:>12,.0f} TL')
print(f'  1000 kaynak projeksiyonu     : {projection_1000:>12,.0f} TL')
print(f'  Optimal esik Recall          : {opt_row.Recall:.3f} (saldiri kaynagi yakalama orani)')
print(f'  Optimal esik Precision       : {opt_row.Precision:.3f}')

print('\n[ONERILER]')
print('  1. Standart %50 esigi yerine finansal olarak optimize esik kullanilmalidir.')
print('  2. SHAP analizine gore en belirleyici ozellikler surekli izlenmelidir.')
print('  3. Model her 30 gunde bir yeni trafik verisiyle guncellenmelidir.')
print('  4. KVKK ihlal olasiligi 4 farkli senaryoda da tasarruf saglamaktadir.')
print('  5. Yuksek riskli protokollere (RDP port 3389, SSH port 22) oncelikli kural tanimi yapilmalidir.')
print('=' * 65)
